# Training Pipeline — Quanvolutional Neural Network

A **Variational Quanvolutional Neural Network**, ported from
[`PlanQK/variational-quanvolutional-neural-networks`](https://github.com/PlanQK/variational-quanvolutional-neural-networks)
(Mattern et al., *Variational Quanvolutional Neural Networks with Enhanced Image Encoding*, 2021).
Like [`qcnn.ipynb`](../quantum/qcnn.ipynb), there is no PyTorch backbone to configure through
`timm` — the original slides a small PennyLane circuit over each image patch as a quantum analogue
of a classical conv kernel, so the load-bearing pieces are ported as plain `torch` tensor ops, the
same departure `qcnn.py` and `vae/models/qvae.py` take from their own PennyLane/Qiskit sources.

**What's ported, in [`cnn/models/quonv.py`](../../models/quonv.py):**
- `Threshold_Encoder` (`encoders/threshold_encoder.py`) — one qubit per pixel of a 2x2 patch,
  `RX(pi)` if the pixel exceeds a threshold else `RX(0)` (identity), applied to `|0>`. Since those
  are the only two angles ever used, the "encoded state" is always exactly one computational basis
  vector — ported here as a direct one-hot lookup rather than simulating the rotation.
- `RandomLayer` / `qml.templates.RandomLayers` (`calculations/random_layer.py`) — the
  variational/random ansatz. Its gate *topology* is fixed by a `calculation_seed`; this port
  replicates PennyLane's (pre-v0.29, matching the paper's 2021 version) exact
  `np.random.seed`-driven gate-placement algorithm bit-for-bit, so the same seed reproduces the
  same circuit, then builds it once as a dense `2^4 x 2^4` real-angle unitary. Only the rotation
  *angles* (`weights`) are what training touches.
- `UniformGateMeasurements` (`measurements/uniform_gate.py`) — `qml.expval(PauliZ)` on every
  qubit, read here directly off `|amplitude|^2` against a precomputed Z-eigenvalue table.
- `QuonvLayer` / `QNNModel` (`models/quonv_layer.py`, `models/qnn.py`) — the 2x2, stride-1,
  no-padding sliding window with **one shared circuit reused at every spatial location** (weight
  sharing, like a classical conv kernel), followed by a single flatten + linear head. The original
  never stacks classical conv/pool layers on top of the quantum one.

**What's replaced:** the original calls its QNode once per `(batch, row, col)` patch in a plain
Python triple loop. Since the circuit is a fixed sequence of unitaries for a given
`calculation_seed`, this notebook instead builds one dense unitary and applies it to every patch of
the batch at once via `torch.unfold` + a single matmul — the same restructuring `qcnn.py` applies
to `takh04/QCNN`.

**The trainable/untrainable ablation is preserved.** The paper's central comparison is trainable
(variational) vs. untrainable (fixed-random) quantum filters — `args.trainable` below toggles
whether the quantum layer's `weights` is a real `nn.Parameter` (gradients flow through the whole
circuit end to end) or a frozen buffer (only the classical `nn.Linear` head trains), matching
`learner.py`'s `requires_grad` gate on `qlayer_1.torch_qlayer.weights`.

No GPU is required — the whole quantum layer is one `16x16` real unitary applied to every 2x2
patch, tiny by CNN standards.

## Install Requirements

Only `torch`/`torchvision` for the data and training loop. No `pennylane` — the circuit is
reimplemented as dense `torch` matrices, so the original's quantum-simulation dependency
(`PennyLane`, `PennyLane-qulacs`, `PennyLane-qiskit`) is never installed.

In [ ]:
!nvidia-smi

In [ ]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

## Get the survey code

The model and handler live in this repository's `survey/src/cnn` package, so the notebook needs a
checkout of it. On Colab it clones into `/content/quantum-quantization`, or `git pull --ff-only`s
that directory if it is already there — so re-running the cell after a push picks up the new code.
Run locally, the notebook already sits inside the repo, so `find_src` climbs to `survey/src` and
git is never touched (your working tree is left alone).

The branch is chosen by *environment*, not by working directory: the imports cell `os.chdir`s into
the checkout, so a cwd-based test would find `cnn` on every re-run and silently skip the pull.

If the repository is private the anonymous clone fails with an authentication error; use a token
URL instead — `REPO_URL = 'https://<GITHUB_TOKEN>@github.com/alexandrachirita98/quantum-quantization.git'`.

In [ ]:
import pathlib
import subprocess
import sys

REPO_URL = 'https://github.com/alexandrachirita98/quantum-quantization.git'

# NB: key off the environment, not the working directory. `os.chdir` in the imports cell moves the
# cwd *inside* the checkout, so a cwd-based test would report "already have the code" on every
# re-run and silently skip the pull.
IN_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').is_dir()
CLONE_DIR = pathlib.Path('/content/quantum-quantization')   # where the Colab checkout goes


def find_src(start):
    """Climb from `start` looking for the survey `src/` root — the directory holding `cnn`."""
    for p in [pathlib.Path(start), *pathlib.Path(start).parents]:
        if (p / 'cnn' / 'models' / 'quonv.py').is_file():
            return p
    return None


if IN_COLAB:
    if CLONE_DIR.exists():                                  # refresh whatever was cloned earlier
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(CLONE_DIR), check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(CLONE_DIR)], check=True)
    SRC = CLONE_DIR / 'survey' / 'src'
else:                                                       # local: the notebook lives in the repo
    SRC = find_src(pathlib.Path.cwd())

assert SRC is not None and (SRC / 'cnn' / 'models' / 'quonv.py').is_file(), f'quonv.py not found under {SRC}'
print('survey src:', SRC)

## Imports

`QNNModel` and the training/data helpers come from the survey's `cnn` package. `SRC` from the
previous cell goes on `sys.path`, and `os.chdir` moves into it so `./data` (the shared
[`src/data`](../../../data) folder) is where `torchvision` downloads MNIST.

In [ ]:
import os
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# import the pipeline from the survey `src/` root, and work from there so './data' resolves inside it
sys.path.insert(0, str(SRC))
os.chdir(SRC)
print('working directory:', os.getcwd())

# drop cached modules, so a `git pull` above is actually reflected on a re-run
for _m in [m for m in list(sys.modules) if m == 'cnn' or m.startswith('cnn.')]:
    del sys.modules[_m]

from cnn.models.quonv import QNNModel
from cnn.handlers.quonv import train_quonv, evaluate, freeze_untrainable

## Configuration

`generate_experiments.py`'s default `hyper_params` for the Threshold-encoder, 2x2-filter
experiment, kept as a plain namespace instead of a YAML sweep file: `img_size=14`
(MNIST downsized from 28x28), filter `2x2` stride `1`, `RandomLayers` with 1 layer / 4 rotations /
`ratio_imprim=0.4`, Adam at `lr=0.01`, batch size 2, 50 epochs capped at 100 steps each.

Set `trainable=False` to reproduce the paper's other ablation arm — a fixed-random quantum filter
with only the classical linear head trained.

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    img_size=14,                                      # MNIST resized from 28x28
    out_features=10,                                  # 10-class digit classification
    filter_size=2,
    stride=1,
    calculation_seed=10,                               # RandomLayers gate-topology seed
    weights_seed=11111,                                # initial rotation-angle seed
    trainable=True,                                   # False = fixed-random quantum filter
    lr=0.01,
    batch_size=2,                                     # matches the original's small batch size
    epochs=5,                                         # paper default is 50; kept small for a demo run
    steps_in_epoch=100,
)

## Device

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using {} device.".format(device))

## Dataset

Full 10-class MNIST (`data/mnist.py`'s default), resized to `img_size x img_size` and left in
`[0, 1]` — z-score normalization is **off** by default in the original, since `Threshold_Encoder`'s
`threshold=0.5` cutoff assumes raw `ToTensor()` pixel values.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Resize(args.img_size)])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = data.DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
test_loader = data.DataLoader(test_dataset, batch_size=64, shuffle=False)

print('train:', len(train_dataset), ' test:', len(test_dataset))

## Model

`QNNModel` builds one `QuonvLayer` (the fixed 2x2 `Threshold_Encoder` + `RandomLayers` + `PauliZ`
circuit, weight-shared across every sliding-window position) followed by `flatten` + a single
`nn.Linear` head — matching the original's flat topology (no classical conv/pool layers).

In [ ]:
net = QNNModel(img_size=args.img_size, out_features=args.out_features,
              filter_size=args.filter_size, stride=args.stride,
              calculation_seed=args.calculation_seed, trainable=args.trainable,
              weights_seed=args.weights_seed).to(device)
freeze_untrainable(net)

print(net)
print('trainable parameters:', sum(p.numel() for p in net.parameters() if p.requires_grad))

## Optimizer & Loss

Adam over `net.parameters()` and `nn.CrossEntropyLoss`, matching `runners.py`. When
`args.trainable=False`, `net.qlayer.weights` is a plain buffer rather than a `Parameter`, so it is
simply absent from `net.parameters()` and only the linear head's weights receive updates —
the same effect as the original's `requires_grad=False` gate.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=args.lr)

## Train

`train_quonv` (from the handler) runs `args.epochs` passes over the training set, capped at
`args.steps_in_epoch` batches per epoch (`Learner.train`'s early `break`), validating on the test
set at the end of each epoch.

In [ ]:
train_quonv(net, optimizer, criterion, train_loader, test_loader, device,
           epochs=args.epochs, steps_in_epoch=args.steps_in_epoch)

## Evaluate

Final test-set accuracy, via the same `evaluate` helper used for validation during training.

In [ ]:
test_acc = evaluate(net, test_loader, device)
print(f'test accuracy: {test_acc:.4f}')